In [ ]:
import sys
import dask
import numpy as np
import dask.array as da
import plotly.graph_objects as go
from scipy.stats import pearsonr
from dask.diagnostics import ProgressBar

sys.path.append('/home/austinbaggetta/csstorage3/CircleTrack/CircleTrackAnalysis')
import plotting_functions as pf

def generate_poisson_spikes(rate, T):
    """
    Generate spike times for a neuron with a constant firing rate using a Poisson process.
    
    Parameters:
    rate (float): Firing rate (spikes per second).
    T (float): Total duration of the simulation (seconds).
    
    Returns:
    spike_times (list): List of spike times.
    """
    spike_times = []
    t = 0
    
    while t < T:
        # Generate the next spike interval
        interval = -np.log(np.random.rand()) / rate
        t += interval
        
        if t < T:
            spike_times.append(t)
    
    return np.asarray(spike_times)

## Set seed for reproducibility
np.random.seed(24601)
nshuffles = 5000
num_spatial_bins = 20
session_length = 60 ## in seconds

In [37]:
## Generate uniformly spiking neuron and occupancy map
neuron = generate_poisson_spikes(rate=5, T=session_length)
time_in_bin = np.random.normal(loc=200, scale=50, size=num_spatial_bins) 
occupancy = time_in_bin / np.sum(time_in_bin)


In [46]:
time_in_bin

array([178.92514516, 150.23149509, 197.8875793 , 216.30926119,
       152.99204868, 168.27794917, 214.13020202, 182.60541051,
       248.28928414, 323.12635992, 194.79316808, 250.40484483,
       236.09501164, 176.99415586, 156.75041751, 122.85917269,
       213.00567496, 197.21640236, 137.12152026, 275.63452885])

In [ ]:
## Create two arrays to correlate
ar = np.random.normal(size=ar_size)
stable_ar = np.random.normal(size=ar_size[1])

In [ ]:
dar = da.from_array(ar, chunks=(30, -1))
dar

In [ ]:
## Faster shuffling with dask?
for shuffle in np.arange(0, nshuffles):
    rolled = shuffle_data(ar)
    cor = return_pearsonr(rolled, stable_ar)
    shuffled_ar = combine_shuffled_r_vals(cor, nshuffles, shuffle)

In [ ]:
shuffled_ar.visualize()

In [17]:
np.arange(0, np.max(sdata['trials'].values)).astype(int)

array([ 0,  1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11, 12, 13, 14, 15, 16,
       17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33,
       34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45])

In [40]:
from numpy.random import RandomState, MT19937, SeedSequence

rs = RandomState(MT19937(SeedSequence(4)))
randomly_sorted = rs.permutation(int(np.max(forward)))
sub_sorted = np.sort(randomly_sorted[:32 + 1])
sub_sorted

array([ 0,  1,  2,  3,  4,  5,  6,  7,  8, 12, 13, 14, 15, 16, 17, 18, 19,
       20, 22, 24, 25, 26, 27, 28, 33, 34, 35, 36, 37, 40, 41, 44, 45])

In [3]:
# %%
import os
import sys
import pandas as pd
import numpy as np
import xarray as xr
from tqdm import tqdm
from os.path import join as pjoin
from natsort import natsorted

sys.path.append('../')
import circletrack_neural as ctn
import circletrack_behavior as ctb
import place_cells as pc

project_dir = 'MultiCon_Imaging'
experiment_dir = 'MultiCon_Imaging5'
mouse_list = ['mc44', 'mc46', 'mc49', 'mc51', 'mc52']
mouse_list = ['mc44']
dpath = f'../../{project_dir}/{experiment_dir}/output/aligned_minian/'
spath = f'../../{project_dir}/{experiment_dir}/output/place_cells_min_trials_end/'
data_type = 'S'
only_running = True
correct_dir = True
smooth = False
binarized = False
velocity_thresh = 10
bin_size = 0.06
nshuffles = 500
percentile = 95
min_event_amount = 0.2
min_trials = 32 ## lowest number of trials a mouse ran on day 16 is 32 trials, which was a Two-context mouse
last_min = False ## set to False if randomize below is True
randomize = True ## used to select forward trials from across the session

for mouse in tqdm(mouse_list):
    mpath = pjoin(dpath, f'{mouse}/{data_type}')
    for session in natsorted(os.listdir(mpath)):
        if (mouse == 'mc56') & (session == f'mc56_{data_type}_1.nc'):
            pass
        else:
            if ('15' not in session) & ('16' not in session):
                pass 
            else:
                print(session)
                save_path = pjoin(spath, f'{mouse}/{data_type}')
                S = xr.open_dataset(pjoin(mpath, session))[data_type]
                sdata = ctn.qc_matrix(S, threshold=True)
                num_neurons = sdata.shape[0]
                minimum_act_bool = pc.minimum_activity_level(sdata, minimum_event_amount=min_event_amount, bin_size_seconds=60, fps=30, func=np.sum, binarized=0)

                if min_trials is not None:
                    forward, reverse = ctb.get_forward_reverse_trials(sdata)

                #     if randomize:
                #         print('help')
                #     else:
                #         if last_min:
                #             sub = forward[-min_trials:]
                #             min_data = sdata[:, sdata['trials'] == sub[0]]
                #             for trial in sub[1:]:
                #                 tdata = sdata[:, sdata['trials'] == trial]
                #                 min_data = xr.concat([min_data, tdata], dim='frame')
                #         else:
                #             min_data = sdata[:, sdata['trials'] == forward[0]]
                #             for trial in forward[1:min_trials]:
                #                 tdata = sdata[:, sdata['trials'] == trial]
                #                 min_data = xr.concat([min_data, tdata], dim='frame')
                # else:
                #     min_data = sdata.copy()

                # if smooth:
                #     smoothed_data = ctn.moving_average_xarray(min_data, ksize=8) 
                # else:
                #     smoothed_data = min_data.copy()

                # neural_data, position_data = ctn.subset_correct_dir_and_running(smoothed_data, correct_dir=correct_dir, only_running=only_running, 
                #                                                                 velocity_thresh=velocity_thresh)


                # ## Calculate observed Skagg's information content and observed spatial coherence
                # population_activity, occupancy, _ = pc.spatial_activity(neural_data, position_data, bin_size=bin_size, binarized=binarized)
                # tuning_curves = pc.get_tuning_curves(population_activity, occupancy)
                # avg_of_avg, spatial_coherence_values = pc.calculate_spatial_coherence(tuning_curves, ksize=8)
                # bits_per_event = pc.skaggs_information_content(tuning_curves, occupancy)
                # first_second = pc.first_second_half_stability(neural_data, bin_size=bin_size)
                # odd_even = pc.odd_even_stability(neural_data, bin_size=bin_size)

                # ## Shuffle data to create a null distribution
                # shuffled_si, shuffled_sc = pc.shuffle_spatial_metrics(neural_data, lin_pos_col='lin_position', bin_size=bin_size, binarized=binarized, nshuffles=nshuffles)
                # shuffled_first_second, shuffled_odd_even = pc.shuffle_stability_metrics(neural_data, bin_size=bin_size, nshuffles=nshuffles)
                
                # place_cells_skaggs = np.array([bits_per_event[neuron] > np.percentile(shuffled_si[:, neuron], percentile) for neuron in np.arange(0, num_neurons)])
                # place_cells_coherence = np.array([spatial_coherence_values[neuron] > np.percentile(shuffled_sc[:, neuron], percentile) for neuron in np.arange(0, num_neurons)])
                # place_cells_first = np.array([first_second[neuron] > np.percentile(shuffled_first_second[:, neuron], percentile) for neuron in np.arange(0, num_neurons)])
                # place_cells_odd = np.array([odd_even[neuron] > np.percentile(shuffled_odd_even[:, neuron], percentile) for neuron in np.arange(0, num_neurons)])

                # sdata = sdata.assign_coords(skaggs_info=('unit_id', bits_per_event),
                #                             skaggs_place=('unit_id', place_cells_skaggs),
                #                             coherence=('unit_id', spatial_coherence_values),
                #                             coherence_fisherz=('unit_id', np.arctanh(spatial_coherence_values)),
                #                             coherence_place=('unit_id', place_cells_coherence),
                #                             first_second=('unit_id', first_second),
                #                             first_second_fisherz=('unit_id', np.arctanh(first_second)),
                #                             first_second_place=('unit_id', place_cells_first),
                #                             odd_even=('unit_id', odd_even),
                #                             odd_even_fisherz=('unit_id', np.arctanh(odd_even)),
                #                             odd_even_place=('unit_id', place_cells_odd),
                #                             shuffled_avg_si=('unit_id', np.mean(shuffled_si, axis=0)),
                #                             shufled_std_si=('unit_id', np.std(shuffled_si, axis=0, ddof=1)),
                #                             minimum_activity_met=('unit_id', minimum_act_bool))
                
                # if not os.path.exists(save_path):
                #     os.makedirs(save_path)
                # sdata.to_netcdf(pjoin(save_path, f'{session}'))


  0%|          | 0/1 [00:00<?, ?it/s]

mc44_S_15.nc
mc44_S_16.nc


  0%|          | 0/1 [00:05<?, ?it/s]


KeyboardInterrupt: 